In [16]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [17]:
df = pd.read_csv('winequality-white.csv')
X = df.drop(columns='quality')
y = df['quality']
df.shape

(4898, 12)

In [18]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.125, random_state=42
)

n = len(X)

## Grid search

In [19]:
def grid_search(make_model, grid, Xtr, ytr, Xva, yva, Xte, yte):
    rows = []
    for params in ParameterGrid(grid):
        m = make_model(**params).fit(Xtr, ytr)
        rows.append({**params,
                     'val_mse': mean_squared_error(yva, m.predict(Xva)),
                     'test_mse': mean_squared_error(yte, m.predict(Xte))})
    return pd.DataFrame(rows).sort_values('val_mse').reset_index(drop=True)

## RandomForestRegressor


In [21]:
rf_grid = {
    'n_estimators': [30, 100],
    'max_depth': [4, 8, 12, 16, 20, None],
    'min_samples_leaf': [1, 2, 4, 8, 16],
    'max_features': [0.2, 0.4, 0.6, 0.8, 1.0],
    'bootstrap': [True, False],
}

t = time.time()
rf_res = grid_search(lambda **p: RandomForestRegressor(random_state=42, n_jobs=-1, **p),
                     rf_grid, X_train, y_train, X_val, y_val, X_test, y_test)

rf_res.head()

,bootstrap,max_depth,max_features,min_samples_leaf,n_estimators,val_mse,test_mse
0,False,20.0,0.2,1,100,0.290606,0.332377
1,False,NaN,0.2,1,100,0.294376,0.329964
2,False,NaN,0.6,2,100,0.298816,0.358940
3,False,20.0,0.6,1,100,0.299028,0.349771
4,False,NaN,0.4,2,30,0.300000,0.352748


## KNeighborsRegressor


In [22]:
scaler = StandardScaler().fit(X_train)
Xtr_s, Xva_s, Xte_s = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)

knn_grid = {
    'n_neighbors': list(range(1, 101)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2, 3],
}

knn_res = grid_search(KNeighborsRegressor, knn_grid, Xtr_s, y_train, Xva_s, y_val, Xte_s, y_test)
knn_res.head()

,n_neighbors,p,weights,val_mse,test_mse
0,25,1,distance,0.298739,0.368550
1,19,1,distance,0.299662,0.367969
2,18,1,distance,0.299994,0.365955
3,28,1,distance,0.300058,0.370215
4,17,1,distance,0.300187,0.365053


## Results: best parameters and test performance

In [23]:
summary = []
for name, res in [('RandomForest', rf_res), ('KNN', knn_res)]:
    best = res.iloc[0]
    params = {k: best[k] for k in res.columns if k not in ('val_mse', 'test_mse')}
    summary.append({'model': name, 'best_params': params,
                    'val_mse (best)': best['val_mse'], 'test_mse (best)': best['test_mse'],
                    'gap (test - val)': best['test_mse'] - best['val_mse']})
summary = pd.DataFrame(summary)
pd.set_option('display.max_colwidth', None)
summary

,model,best_params,val_mse (best),test_mse (best),gap (test - val)
0,RandomForest,"{'bootstrap': False, 'max_depth': 20.0, 'max_features': 0.2, 'min_samples_leaf': 1, 'n_estimators': 100}",0.290606,0.332377,0.041770
1,KNN,"{'n_neighbors': 25, 'p': 1, 'weights': 'distance'}",0.298739,0.368550,0.069811


## Relationship between validation risk and test risk

Random forest does better on the test set.

For both models, test MSE is higher than validation MSE for the chosen parameters. This is expected. Since the parameters are selected by evaluation using the validation test, the noise in the validation set affected the model (bias).

However, validation and test risks are still strongly correlated, by observing the test errors.

Validation risk is a good tool for *choosing* hyperparameters but a biased (optimistic) estimate of performance once it has been minimized over many candidates. The more parameter sets we try relative to the validation set size, the larger this optimism can be. This is why the final performance must be reported on a held-out test set that played no role in model selection.
